# Cookie Cats A/B Test — Gate Placement & Player Retention

**Context.** Cookie Cats is a mobile puzzle game with a progression gate that pauses play. This experiment randomly assigned new players to a gate at **level 30** (`gate_30`, control) or **level 40** (`gate_40`, treatment) and measured whether moving it later affected retention.

**Question.** Does moving the gate to level 40 improve day-1 or day-7 retention?

**Method.** Understand the data → clean → randomization check → retention rates → two-proportion z-test (+ chi-square) → bootstrap 95% CI → recommendation.


First, load the libraries and the dataset. `kagglehub` downloads the file to a local cache, so the CSV never has to be committed to the repo.


In [1]:
import kagglehub, os, glob
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency

path = kagglehub.dataset_download("yufengsui/mobile-games-ab-testing")
df = pd.read_csv(glob.glob(os.path.join(path, "*.csv"))[0])


## 1. Understand the dataset

Before analysing anything, I look at what the data contains. Each **row** is one player who installed the game during the experiment. Let's see the raw table first, then explain the columns.


In [2]:
print("Shape:", df.shape)
df.head()


Shape: (90189, 5)


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


Now that we've seen real rows, here is what each column means:

| Column | Meaning |
|---|---|
| `userid` | Unique player id (label only). |
| `version` | Experiment group: `gate_30` (control) or `gate_40` (treatment). |
| `sum_gamerounds` | Rounds played in the first 14 days (engagement). |
| `retention_1` | Came back 1 day after install? (True/False) |
| `retention_7` | Came back 7 days after install? (True/False) — **primary metric**. |

The retention columns are boolean, so their **mean** is directly the retention rate.


Next I check the data types and confirm there are no missing values, using `.info()`.


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.8 MB


All 5 columns have 90,189 non-null values, so **nothing is missing**. The types make sense: `version` is text (the group label), the two retention columns are boolean (handy — their mean is a rate), and `sum_gamerounds` is a whole number.


Now a numeric summary of `sum_gamerounds` (the only numeric column worth investigating — `userid` is numeric too but it's just an identifier, so its statistics are meaningless).


In [4]:
print(df["sum_gamerounds"].describe())


count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64


The median is 16 rounds, but the maximum is 49,854. That one value is far outside anything realistic for 14 days of play, so it's almost certainly a bot or a logging error. I look at it in the cleaning step rather than leaving it in the averages.

## 2. Clean and prepare

Cleaning here is light but each check has a clear reason. I will:

1. confirm there are **no duplicate players** (a repeated `userid` would double-count someone, or worse put them in both groups),
2. decide what to do with the **extreme outlier** seen above,
3. confirm the **group labels** are exactly the two we expect (a typo like `gate_35` would be a warning sign).

Column names are already clear, so no renaming is needed.


**2a. Duplicate check.** Every player should appear once. The assert passes silently if there are zero duplicates and stops the notebook if not.


In [5]:
assert df["userid"].duplicated().sum() == 0, "Duplicate userids found"
print("No duplicate players.")


No duplicate players.


**2b. Outlier decision.** I only drop values that are physically impossible. Around 50,000 rounds in 14 days works out to thousands per day, which isn't a real player, so I treat it as corrupted data.

I set the cutoff high, at 10,000, on purpose. My first instinct was to cut at something like 1,400 (about 100 rounds a day), but that would throw out players who are unusual without being impossible, and dropping real players changes who's actually in the experiment. So the rule is to remove the impossible and keep the merely extreme. It barely affects the result either way, since the outcome we care about is retention, not rounds played.

In [6]:
before = len(df)
df = df[df["sum_gamerounds"] < 10000].copy()
print(f"Rows removed as implausible outliers: {before - len(df)}")


Rows removed as implausible outliers: 1


As expected, this removes just **1 row**, the single corrupted record, leaving the analysis essentially untouched.


**2c. Group labels.** Confirm `version` contains exactly the two expected groups and nothing unexpected.


In [7]:
print("Groups:", list(df["version"].unique()))


Groups: ['gate_30', 'gate_40']


Exactly the two groups we expect (`gate_30`, `gate_40`), no typos or stray labels. The data is ready to analyse.


## 3. Randomization balance

In a valid A/B test the random assignment should split players roughly evenly. Very unequal group sizes could signal a problem in how the experiment was run. I check the share of players in each group.


In [8]:
(df["version"].value_counts(normalize=True) * 100).round(1)


version
gate_40    50.4
gate_30    49.6
Name: proportion, dtype: float64

The split is close to 50/50 (49.6% vs 50.4%), consistent with proper random assignment. Good to proceed.


## 4. Retention rates by group

Now the core comparison. Retention rate = the share of players who came back, which is just the **mean** of the True/False retention column. We compute it per group for both horizons, then isolate the **day-7 difference**. The primary metric is the day-7 retention, since returning a week after install is a stronger signal of lasting interest than returning after one day.

In [9]:
summary = df.groupby("version").agg(
    users=("userid", "count"),
    retention_1=("retention_1", "mean"),
    retention_7=("retention_7", "mean"),
)
summary[["retention_1", "retention_7"]] = (summary[["retention_1", "retention_7"]] * 100).round(2)
print(summary, "\n")

# Isolate the day-7 difference (the number the significance test will assess)
r30 = df.loc[df["version"] == "gate_30", "retention_7"]
r40 = df.loc[df["version"] == "gate_40", "retention_7"]
p30, p40 = r30.mean(), r40.mean()

print(f"gate_30: {p30*100:.2f}%   gate_40: {p40*100:.2f}%")
print(f"Difference: {(p30 - p40)*100:.2f} pp   Relative lift (gate_30): {(p30/p40 - 1)*100:.2f}%")

         users  retention_1  retention_7
version                                 
gate_30  44699        44.82        19.02
gate_40  45489        44.23        18.20 

gate_30: 19.02%   gate_40: 18.20%
Difference: 0.82 pp   Relative lift (gate_30): 4.50%


gate_30 comes out ahead on both metrics. At day 7 the gap is 0.82 percentage points (19.02% vs 18.20%), or 4.5% in relative terms.

That gives the direction and size of the difference, but not whether it's real. With ~45,000 players per group, two groups will almost never land on exactly the same rate even if the gate change does nothing, so a small gap is expected by chance alone. The z-test in section 5 checks whether this gap is bigger than that chance variation.

## 5. Significance — two-proportion z-test

The difference is 0.82 pp. The question is whether a gap that size could show up by chance even if the gate change had no effect.

The hypotheses:
- H0: both groups have the same true day-7 retention; the observed gap is chance.
- H1: the retention rates differ.

The two-proportion z-test takes the gap between the two rates and divides it by the amount of random variation expected at these sample sizes. That ratio is the z-statistic, and it converts to a p-value: the probability of a gap this large if H0 were true. Below 0.05, I treat the gap as unlikely to be chance.

I also run a chi-square test on the same table as a cross-check. It answers the same question a different way, so agreement between the two is reassuring.

In [10]:
successes = np.array([r30.sum(), r40.sum()])      # players who returned in each group
n_obs     = np.array([r30.count(), r40.count()])  # total players in each group
z_stat, p_value = proportions_ztest(successes, n_obs)

chi2, chi_p, *_ = chi2_contingency(pd.crosstab(df["version"], df["retention_7"]))

print(f"z = {z_stat:.3f}   p = {p_value:.4f}")
print(f"chi-square p = {chi_p:.4f}")
print("Significant at 0.05." if p_value < 0.05 else "Not significant at 0.05.")


z = 3.157   p = 0.0016
chi-square p = 0.0016
Significant at 0.05.


z = 3.16 and p = 0.0016. The chi-square test gives the same p-value, which is a good sign that the result doesn't depend on the choice of test.

Since p is well below 0.05, I reject the null hypothesis of equal retention. The difference between the two groups is very unlikely to be chance, and from section 4 it favours gate_30.

On reading the z-value: under the null hypothesis, z follows a standard normal distribution (mean 0, standard deviation 1), where about 95% of values fall between -2 and +2 and 99.7% between -3 and +3. A z of 3.16 is outside that range, which is why the p-value is so small. The large sample is what makes it possible to detect a difference this small: 0.82 pp is tiny, but with 45,000 players per group the standard error is small too, so the gap still lands far out in the tail.

## 6. Effect size — bootstrap 95% confidence interval

The z-test says the difference is real but not how precisely it's measured. A confidence interval gives a plausible range for the true difference instead of a single number.

The bootstrap gets there without a formula. Treat the sample as if it were the whole population, then draw a new sample of the same size from it, with replacement, and recompute the difference. Repeat 2,000 times. The spread of those 2,000 values reflects how much the difference would move from one sample to another, and the middle 95% of them is the confidence interval.

In [11]:
rng = np.random.default_rng(42)
a, b = r30.values, r40.values
diffs = np.array([rng.choice(a, len(a), replace=True).mean()
                  - rng.choice(b, len(b), replace=True).mean() for _ in range(2000)])
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5]) * 100

print(f"Observed difference: {(p30 - p40)*100:.2f} pp")
print(f"95% CI: [{ci_low:.2f} pp, {ci_high:.2f} pp]")


Observed difference: 0.82 pp
95% CI: [0.33 pp, 1.35 pp]


The single best estimate of the difference is 0.82 pp, and the 95% CI runs from 0.33 to 1.35 pp. The interval matters more than the single number here, because it shows how precisely the difference is pinned down.

The whole interval sits above zero, so gate_30's advantage holds across the resampling and never flips to gate_40. That matches the z-test. It also shows the effect is small: even the top end is only about 1.35 pp.

**Recommendation: keep the gate at level 30.**

Moving it to level 40 did not improve retention. Day-7 retention was higher for gate_30 (19.0% vs 18.2%), the difference is statistically significant (p = 0.0016), and the 95% CI [0.33, 1.35] pp stays above zero. With ~45,000 players per group the test is well powered, so this is a real difference rather than noise, but it's a small one, so I wouldn't expect a large jump in retention from keeping the gate at 30.

The main limitation is that this measures retention, not money. Day-7 retention is a reasonable proxy for long-term engagement, but a follow-up looking at in-app spend or lifetime value would be needed to confirm the financial impact.